# Session 13 · Models & Generalisation
### Linear and logistic regression, decision trees — underfitting or overfitting?

**Data Mining 2026/27** · doc. dr. Uroš Godnov · Mon 4 Jan / Wed 6 Jan 2027 · 3 × 45 min + 15 min break

Welcome back, and happy New Year! With JJ you saw that models of varying complexity trade off bias and variance. Today you **see it happen in code**: you train the classic models, measure them honestly, and learn the toolkit every data miner uses to tell **underfitting** from **overfitting** — cross-validation, learning curves, validation curves, regularisation and grid search.

> 💾 **File → Save a copy in Drive** before you start.

### 🎯 Minimum path — new to Python, behind, or working on your own?
Run the notebook from top to bottom (**Shift + Enter**) and put your attention where the homework and the project need it:
- 📖 **Study closely:** A3 logistic regression · A4 decision trees and overfitting · A5 the generalisation toolkit · A6 the assignment briefing.
- ▶️ **Just run, then read the "So what?":** A1 linear regression · A2 Ridge and Lasso.
- ✏️ **Part B:** Tasks 1–3; Tasks 4–5 take about a minute each, so do them if time allows.
- ⏩ **Skip without guilt today:** the 🚀 side quests and Part C, the 📈/💻 boxes, the 🧠 think questions, and every 🧪 *Try it* after the first one.

<details>
<summary><b>ℹ️ Session details</b> — timing, learning objectives, icons, data and the two lanes (click to open)</summary>

| Part | Time | What we do |
|---|---|---|
| Warm-up | 5 min | recall S12 · train 99 %, test 70 % — what happened? |
| **A — Guided walkthrough** | 80 min | linear regression · Ridge & Lasso · logistic regression · decision trees · the generalisation toolkit · assignment briefing |
| ☕ Break | 15 min | |
| **B — Your turn** | 35 min | bank telemarketing: baseline, logistic regression, tuned trees, diagnosis |
| Wrap-up | 10 min | key takeaways, exit ticket, what's next |
| 🚀 **C — Advanced track (optional)** | self-paced, ≈ 105 min | under the hood: logistic regression from scratch, bias–variance by simulation, nested CV, a corrected t-test, calibration, tests for ML code |

**Learning objectives** — after this session you can:
1. train and evaluate linear regression (MAE, RMSE, R²) and read its residuals;
2. explain how Ridge and Lasso prevent overfitting, and use Lasso as feature selection;
3. train logistic regression and decision trees for classification, and interpret odds ratios and tree rules;
4. diagnose underfitting vs overfitting with train/validation scores, learning curves and validation curves;
5. use k-fold cross-validation and `GridSearchCV` correctly, inside a pipeline.

| Icon | Meaning |
|---|---|
| 🔮 | **Predict before you run** |
| 🧪 | **Try it** — 2–5 minutes during the walkthrough |
| ✏️ | **TODO** — your task in Part B |
| ✅ | **Self-check** (⏳ = not attempted yet) |
| 🛟 | **Rescue cell** — run only if stuck |
| 🤖 | **AI rule** — try first; then Gemini may *explain* or *hint* |
| 🚀 | **Fast lane** — optional side quests (Part A) and the Advanced track (Part C); never graded |
| 📈 💻 | **Two lenses** — a one-minute link to what you already know: 📈 economics and statistics, 💻 programming and databases |

**Data today** (from the web): Ames Housing (De Cock, 2011) · Telco Customer Churn (IBM) · Bank Marketing (Moro, Cortez & Rita, 2014) · Energy Efficiency (Tsanas & Xifara, 2012). Part C also simulates its own data.

### 🧭 Two lanes — pick yours

| Lane | For whom | How to work through this notebook |
|---|---|---|
| 🟢 **Core** | everyone — especially if Python is still new to you | Part A with the lecturer → Part B → wrap-up. This is everything the assignment and the project need. |
| 🚀 **Fast lane** | you already write Python comfortably (functions, loops, a bit of pandas) — e.g. you passed the 🚀 check in S00 | Follow Part A, and while we cover basics you know, work on the 🚀 **side quests**. Do Part B quickly (it is still the core of the project), then go on to **Part C · 🚀 Advanced track** after the wrap-up. |

You can switch lanes at any time. Everything marked 🚀 is optional, never graded, and its solutions are in the solutions notebook. Use the 📑 table of contents (left sidebar) to jump between parts.

**Two lenses.** 📈 boxes link the topic to economics and statistics, 💻 boxes to programming and databases. Read the one for your programme, or both. In Part C, 📈 challenges need more statistics and less code; 💻 challenges need more programming. **Strong in statistics but new to Python?** Stay in the 🟢 core for the code, and after the wrap-up try the 🧠 think questions and the 📈 challenges.

</details>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/dm-2627-python/blob/main/notebooks/s13-models-generalisation.ipynb)

## 0 · Warm-up (5 min)

### 🔁 Recall from last session (3 min)

Answer **from memory** first — no scrolling back, no AI. Then run the cell below and compare.

1. **Split first.** Name one *stateful* preparation step. On which data is it fitted — and why does it belong *inside* the pipeline?
2. `PoolQC` is empty for most Ames houses. Impute it — or what? And how do we encode an ordered rating such as `KitchenQual` (Po < Fa < TA < Gd < Ex)?
3. A colleague's cancellation model reaches a cross-validated ROC-AUC of **1.0**. Celebrate or worry — what do you check first?

In [ ]:
#@title 👀 Show the answers { display-mode: "form" }
print("1. A median imputer, a scaler or a one-hot encoder: each learns something from data (a median, a mean and SD, a list of categories), so it is fitted on the training data only. Inside the pipeline it is re-fitted on every training fold, so cross-validation never peeks at the validation fold. Today's CV and grid search rely on exactly that.")
print('2. Do not impute: an empty PoolQC means \'no pool\' (structural), so it becomes the category "None". Ordered ratings go through an OrdinalEncoder with the order written out (None = 0 … Ex = 5); categories without an order, such as Neighborhood, are one-hot encoded.')
print('3. Worry. A perfect score is a red flag for leakage: look for a feature that encodes the outcome or is only known afterwards. In S12, reservation_status pushed the AUC above 0.99.')

> 🔮 **Predict before you run:** A model scores **99 % accuracy on the training data** but **70 % on the test data**. Underfitting or overfitting — and what would you try first?

## ⚙️ Setup

In [ ]:
import io
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LassoCV, LinearRegression, LogisticRegression, Ridge, RidgeCV
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             mean_absolute_error, r2_score, roc_auc_score, root_mean_squared_error)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_score, learning_curve,
                                     train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
#@title 🧰 Course helpers — run this cell (no need to read it) { display-mode: "form" }
CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

def check(label, test, hint=""):
    """Give instant feedback on an exercise without ever stopping 'Run all'.

    `test` is a function (usually a lambda) that returns True when the answer is right.
    """
    try:
        ok = bool(test())
    except NameError as err:
        CHECK_RESULTS[label] = None
        print(f"⏳ {label}: not attempted yet ({err}).")
        return
    except Exception as err:  # wrong type, wrong shape, missing column, ...
        CHECK_RESULTS[label] = False
        print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
        if hint:
            print(f"   💡 Hint: {hint}")
        return
    CHECK_RESULTS[label] = ok
    if ok:
        print(f"✅ {label}: correct!")
    else:
        print(f"❌ {label}: not yet.")
        if hint:
            print(f"   💡 Hint: {hint}")

def progress(*, return_count=False):
    """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
    core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
    fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
    solved = sum(1 for v in core if v)
    print(f"Solved {solved} of {len(core)} exercises checked so far.")
    if fast:
        print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
    for label, v in CHECK_RESULTS.items():
        print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
    if return_count:
        return solved

print("Helpers loaded ✔")

In [ ]:
#@title 🧰 S12 recap — run me: Ames data, split, outlier rule, features and the preprocessing pipeline { display-mode: "form" }
QUALITY = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ORDINAL_COLS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]

def add_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    return df

ames = fetch_openml(data_id=42165, as_frame=True).frame.drop(columns="Id")
train, test = train_test_split(ames, test_size=0.2, random_state=42)
train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300_000))]      # the two partial sales
train, test = add_features(train), add_features(test)
X_train, y_train = train.drop(columns="SalePrice"), np.log1p(train["SalePrice"])
X_test, y_test = test.drop(columns="SalePrice"), np.log1p(test["SalePrice"])
NOMINAL_COLS = [c for c in X_train.select_dtypes(exclude="number").columns if c not in ORDINAL_COLS]
NUMERIC_COLS = list(X_train.select_dtypes("number").columns)

def make_ames_preprocess():
    numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
    ordinal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                        ("encode", OrdinalEncoder(categories=[QUALITY] * len(ORDINAL_COLS),
                                                  handle_unknown="use_encoded_value", unknown_value=-1))])
    nominal = Pipeline([("fill_none", SimpleImputer(strategy="constant", fill_value="None")),
                        ("encode", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", numeric, NUMERIC_COLS), ("ord", ordinal, ORDINAL_COLS),
                              ("nom", nominal, NOMINAL_COLS)])

print("Ames ready — train:", X_train.shape, " test:", X_test.shape)

---
# Part A · Guided walkthrough

## A1 · Linear regression (15 min)

**Linear regression** predicts the target as a weighted sum: *ŷ = b₀ + b₁x₁ + b₂x₂ + …*. *Ordinary least squares* (OLS) picks the weights that minimise the squared errors on the training data. Because we model **log** prices, a coefficient of 0.05 means roughly **+5 % price**.

The recap cell above rebuilt everything from S12: the split, the outlier rule, the engineered features and `make_ames_preprocess()`.

### 🗺️ You already know this

| I want to … | scikit-learn / statsmodels | Stata | Excel | What changes in ML |
|---|---|---|---|---|
| fit a linear regression | `LinearRegression().fit(X, y)` · `smf.ols("y ~ x1 + x2", data=df).fit()` | `regress y x1 x2` | `=LINEST(y, X, TRUE, TRUE)` · Analysis ToolPak → Regression | judged by the error on **new** data (test MAE, RMSE), not by in-sample R² or t-statistics |
| robust standard errors | `smf.ols(...).fit(cov_type="HC1")` | `regress y x1 x2, vce(robust)` | — | standard errors serve inference; scikit-learn reports none, because models are chosen by validation error |
| fit a logistic regression | `LogisticRegression().fit(X, y)` · `smf.logit("y ~ x1 + x2", data=df).fit()` | `logit y x1 x2` · `logistic y x1 x2` (odds ratios) | — (only via Solver) | scikit-learn penalises by default (`C=1.0`): the coefficients are shrunk towards 0 |
| categorical predictors | `OneHotEncoder(handle_unknown="ignore")` in the pipeline · `C(region)` in a formula | `i.region` | dummy columns by hand | the encoder learns its categories from the training data and must survive new ones |
| predictions | `model.predict(X_new)` · `model.predict_proba(X_new)[:, 1]` | `predict yhat` (after `logit`: the probability) | `=TREND(y, X, X_new)` | the point is predicting rows the model has **not** seen |
| hold out a test set | `train_test_split(X, y, test_size=0.2, random_state=42)` | `splitsample, generate(sample) split(0.8 0.2) rseed(42)` | a `=RAND()` column, sort, cut | the test set is used **once**; all choices are made with CV on the training part |
| choose the predictors | `LassoCV(cv=5)` · `GridSearchCV(...)` | `lasso linear y x1-x50` · `stepwise, pr(.05): regress y x1-x50` | — | chosen by cross-validated error, not by significance — and the chosen set is not causal |
| ROC-AUC of a classifier | `roc_auc_score(y_test, p_test)` | `lroc` (estimation sample) · `roctab y p if sample == 2` (held-out rows) | — | always on held-out rows: the in-sample AUC flatters the model |

Same models, different question: econometrics asks *what is the effect?*, machine learning asks *how well does it predict new cases?* (`splitsample` and `lasso` need Stata 16 or newer.)

In [ ]:
ols = Pipeline([("prep", make_ames_preprocess()), ("reg", LinearRegression())]).fit(X_train, y_train)
pred = ols.predict(X_test)

print(f"R² (log prices):  train {r2_score(y_train, ols.predict(X_train)):.3f}   test {r2_score(y_test, pred):.3f}")
print(f"MAE  (USD): {mean_absolute_error(np.expm1(y_test), np.expm1(pred)):,.0f}")
print(f"RMSE (USD): {root_mean_squared_error(np.expm1(y_test), np.expm1(pred)):,.0f}")

| Metric | Meaning | Use it when |
|---|---|---|
| **MAE** | average absolute error, in the target's unit (USD) | you need a number managers understand |
| **RMSE** | like MAE, but large errors count more | big misses are especially costly |
| **R²** | share of the variance explained (1 = perfect, 0 = no better than predicting the mean) | comparing models on the same data |

**So what?** The test R² (0.92) is close to the training R² (0.95), so the model hardly overfits. A typical miss of about **$15,000** is roughly 10 % of the median test price ($154,000).

> ⚠️ **Common mistake:** computing MAE or RMSE on the **log** scale and reporting it as dollars. The model predicts log prices, so back-transform with `np.expm1` (the inverse of `np.log1p`) first — as the cell above does.

> 📈 **Economist's view.** Econometrics judges OLS by its coefficients, standard errors and p-values — *inference* about how the data were generated. Here the same estimator is judged only by its error on houses it has never seen — *prediction* — so a model can predict well although none of its coefficients is causal.

> 💻 **Informatics view.** Treat the fitted pipeline as a build artefact: the same code, data and seed must give the same model, so you version it, test it (🚀 C6) and ship the whole pipeline rather than a list of coefficients. The test error is its acceptance test.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(pred, y_test - pred, s=10, alpha=0.6)
ax.axhline(0, color="red")
ax.set_xlabel("predicted log price (log USD)")
ax.set_ylabel("residual (actual − predicted)")
ax.set_title("Residuals without a clear pattern: the linear model fits reasonably")
plt.show()

In [ ]:
# 🚀 Side quest: OLS by hand. Least squares has a closed-form solution: the weights b minimise ‖A·b − y‖²,
#    where A is the feature matrix with a column of ones in front (for the intercept). For
#    SQ_COLS = ["GrLivArea", "TotalSF", "OverallQual", "HouseAge", "TotalBath"], build A from X_train,
#    solve with np.linalg.lstsq and store the 6 weights (intercept first) in `beta`. Compare them with
#    LinearRegression().fit(X_train[SQ_COLS], y_train) and turn the coefficients into % price per unit.
#    Bonus: why is lstsq safer than the textbook formula (AᵀA)⁻¹Aᵀy?
#    Compare np.linalg.cond(A) with np.linalg.cond(A.T @ A).
# SQ_COLS = ["GrLivArea", "TotalSF", "OverallQual", "HouseAge", "TotalBath"]
# A = ...                                  # n × 6: a column of ones, then the five features
# beta, *_ = np.linalg.lstsq(A, ..., rcond=None)
# ols5 = LinearRegression().fit(...)       # compare beta with ols5.intercept_ and ols5.coef_


In [ ]:
#@title ✅ Check — 🚀 A1 side quest { display-mode: "form" }
check('🚀 A1 side quest', lambda: np.allclose(beta, np.r_[LinearRegression().fit(X_train[SQ_COLS], y_train).intercept_, LinearRegression().fit(X_train[SQ_COLS], y_train).coef_]), hint='A = np.column_stack([np.ones(len(X_train)), X_train[SQ_COLS]]) — the intercept is beta[0].')

## A2 · Regularisation: Ridge and Lasso (15 min)

OLS chases the training data. With **many features and few rows** it starts memorising noise. Regularisation adds a penalty for large coefficients:
- **Ridge (L2)** shrinks all coefficients towards zero;
- **Lasso (L1)** shrinks them and sets many **exactly to zero**, which makes it a feature-selection method too;
- **alpha** sets the strength of the penalty (AI Studio: the *ridge* parameter of Linear Regression).

> 🔮 **Predict before you run:** We train OLS, Ridge and Lasso on only **200 houses** — fewer rows than the 231 columns the pipeline creates from them. What happens to OLS on the test set?

In [ ]:
small = X_train.sample(200, random_state=1)
y_small = y_train.loc[small.index]
for name, reg in [("OLS", LinearRegression()),
                  ("Ridge (alpha=10)", Ridge(alpha=10)),
                  ("Lasso (alpha=0.001)", Lasso(alpha=0.001, max_iter=50_000))]:
    m = Pipeline([("prep", make_ames_preprocess()), ("reg", reg)]).fit(small, y_small)
    print(f"{name:20s} train R² {r2_score(y_small, m.predict(small)):.3f}   test R² {r2_score(y_test, m.predict(X_test)):.3f}")

**So what?** With more columns than rows, OLS can fit the 200 houses almost perfectly (training R² ≈ 1) — and fails on new houses. The penalised models give up a little training fit and generalise far better.

In [ ]:
ridge = Pipeline([("prep", make_ames_preprocess()), ("reg", Ridge(alpha=10))]).fit(X_train, y_train)
coefs = pd.Series(ridge.named_steps["reg"].coef_, index=ridge.named_steps["prep"].get_feature_names_out())
pd.concat([coefs.nlargest(5), coefs.nsmallest(5)]).round(3)

**Reading coefficients** (log target): `Neighborhood_Crawfor` +0.08 means roughly **8 % more** than a comparable house in an average neighbourhood. Numeric features are standardised, so their coefficients are *per standard deviation*. And remember: coefficients describe associations in this data, not causes.

In [ ]:
rows = []
for a in np.logspace(-4, -0.5, 12):
    m = Pipeline([("prep", make_ames_preprocess()), ("reg", Lasso(alpha=a, max_iter=20_000))]).fit(X_train, y_train)
    rows.append({"alpha": a, "features kept": int((m.named_steps["reg"].coef_ != 0).sum()),
                 "test R²": r2_score(y_test, m.predict(X_test))})
path = pd.DataFrame(rows)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.semilogx(path["alpha"], path["features kept"], marker="o")
ax1.set_xlabel("alpha (strength of the penalty, log scale)")
ax1.set_ylabel("features kept", color="tab:blue")
ax2 = ax1.twinx()
ax2.semilogx(path["alpha"], path["test R²"], marker="s", color="tab:red")
ax2.set_ylabel("test R²", color="tab:red")
ax1.set_title("Lasso: a stronger penalty keeps fewer features — up to a point, at little cost")
plt.show()
path.round(3)

About 25 features already give a test R² near 0.90, compared with 0.92 when (almost) all are kept — a much simpler model for little loss. (We peeked at the test set only to draw the picture. To **choose** alpha, use cross-validation, as `LassoCV` does.)

> 📈 **Economist's view.** Lasso selects *predictors*, not causes: of two correlated variables it keeps one more or less by chance, and a variable it drops can still matter causally. For causal effects with many controls, econometrics uses double-selection Lasso (Belloni, Chernozhukov & Hansen, 2014; in Stata `dsregress`).

> 💻 **Informatics view.** Lasso is feature selection built into training: every input it drops is one column fewer to collect, validate and monitor in production.

In [ ]:
# 🧪 Try it: Fit a pipeline `lasso_cv` with `make_ames_preprocess()` and LassoCV(cv=5, random_state=42,
#    max_iter=20_000), which picks alpha by cross-validation. Store the number of non-zero
#    coefficients in `n_kept` and print the chosen alpha and the test R².
# lasso_cv = Pipeline([...]).fit(X_train, y_train)
# n_kept = ...


In [ ]:
#@title ✅ Check — A2 Try it { display-mode: "form" }
check('A2 Try it', lambda: 70 <= n_kept <= 140, hint='Count the non-zero entries of lasso_cv.named_steps["reg"].coef_.')

In [ ]:
# 🚀 Side quest: The whole Lasso path in one call. lasso_path (from sklearn.linear_model) solves the
#    Lasso for a whole list of alphas, starting each fit from the previous solution — much faster than
#    separate fits. It fits no intercept, so centre the data first: fit make_ames_preprocess() on X_train,
#    transform X_train into a dense array and subtract the column means (`Xc`); centre y_train the same
#    way (`yc`). Then run
#    alphas_lp, coefs_lp, _ = lasso_path(Xc, yc, alphas=np.logspace(-0.5, -4, 30), max_iter=20_000).
#    Store the name of the feature that enters FIRST (non-zero at the largest alpha) in `first_in` and
#    plot the coefficient paths of the first eight. Then think: is `first_in` really the most important feature?
# from sklearn.linear_model import lasso_path
# prep_lp = make_ames_preprocess().fit(X_train)
# Xp = ...                                 # dense array of the transformed X_train
# Xc, yc = ..., ...                        # centred
# alphas_lp, coefs_lp, _ = lasso_path(Xc, yc, alphas=np.logspace(-0.5, -4, 30), max_iter=20_000)
# first_in = ...


In [ ]:
#@title ✅ Check — 🚀 A2 side quest { display-mode: "form" }
check('🚀 A2 side quest', lambda: first_in == "ord__FireplaceQu" and coefs_lp.shape[0] == 268, hint='Centre both X and y; alphas_lp runs from large to small, so the first feature to enter is non-zero earliest.')

## A3 · Logistic regression — Telco churn (15 min)

Logistic regression computes a linear score and squeezes it through the **sigmoid** function into a probability between 0 and 1; by default, a probability above 0.5 means "yes". We reuse the Telco data from S10.

In [ ]:
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(TELCO_URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)   # S10: not billed yet
y_tel = (telco["Churn"] == "Yes").astype(int)
X_tel = telco.drop(columns=["customerID", "Churn"])

NUM_TEL = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CAT_TEL = [c for c in X_tel.columns if c not in NUM_TEL]
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_tel, y_tel, test_size=0.2, stratify=y_tel, random_state=42)

def make_telco_preprocess():
    return ColumnTransformer([("num", StandardScaler(), NUM_TEL),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_TEL)])

print("train:", Xt_train.shape, "| churn rate:", round(yt_train.mean(), 3))

In [ ]:
logreg = Pipeline([("prep", make_telco_preprocess()), ("clf", LogisticRegression(max_iter=1000))]).fit(Xt_train, yt_train)
proba = logreg.predict_proba(Xt_test)[:, 1]
print("first five churn probabilities:", proba[:5].round(2))
print(f"test ROC-AUC: {roc_auc_score(yt_test, proba):.3f}\n")
print(classification_report(yt_test, logreg.predict(Xt_test), target_names=["stays", "churns"], digits=3))

**So what?** Accuracy 0.81 sounds fine, but the model catches only **56 %** of the churners (recall), and **66 %** of the customers it flags really churn (precision). ROC-AUC (0.84) judges the *ranking* over all thresholds; S14 shows how to choose the threshold from business costs.

> ⚠️ **Common mistake:** `roc_auc_score(y_test, model.predict(X_test))`. `predict` returns hard 0/1 labels, so the AUC is computed from a single threshold and comes out too low (here 0.73 instead of 0.84). ROC-AUC needs scores: pass `model.predict_proba(X_test)[:, 1]`.

In [ ]:
ConfusionMatrixDisplay.from_estimator(logreg, Xt_test, yt_test, display_labels=["stays", "churns"], cmap="Blues")
plt.title("Logistic regression — Telco test set")
plt.show()

> 🔮 **Predict before you run:** Which contract type lowers the **odds of churn** the most — and which customer feature raises them the most?

In [ ]:
odds = pd.Series(np.exp(logreg.named_steps["clf"].coef_[0]), index=logreg.named_steps["prep"].get_feature_names_out())
pd.concat([odds.nsmallest(5), odds.nlargest(5)]).round(2)

An **odds ratio** (OR) multiplies the odds of churning, everything else held equal: OR < 1 protects, OR > 1 raises the risk. Two details matter here:
- Every category has its **own** 0/1 column and the penalty shrinks them all, so each OR is measured against an "average" customer. **Compare categories with each other:** two-year vs month-to-month = 0.46 / 1.79 ≈ 0.26.
- Numeric features are standardised, so their OR is *per standard deviation*: `tenure` 0.28 per SD (about two years) — loyalty protects. But `tenure`, `MonthlyCharges` and `TotalCharges` move together (TotalCharges ≈ tenure × MonthlyCharges), so "everything else held equal" is partly fiction for these three. That is why `TotalCharges` (1.71) and `MonthlyCharges` (0.55) look odd: read them as a group, not one by one.

In [ ]:
# 🚀 Side quest: Odds ratios with 95 % confidence intervals. scikit-learn reports no standard errors;
#    statsmodels does. Fit an UNPENALISED logistic regression with the formula API
#    (import statsmodels.formula.api as smf):
#    smf.logit("churn ~ I(tenure / 12) + MonthlyCharges + C(Contract) + C(InternetService) + SeniorCitizen",
#    data=Xt_train.assign(churn=yt_train)).fit(disp=0). Store a DataFrame `or_table` with the columns "OR",
#    "2.5 %" and "97.5 %" (exponentiate .params and .conf_int()). What is the odds ratio of a two-year
#    contract now — and why does it differ from the 0.46 above?
# import statsmodels.formula.api as smf
# logit_res = smf.logit("churn ~ ...", data=Xt_train.assign(churn=yt_train)).fit(disp=0)
# ci = np.exp(logit_res.conf_int())
# or_table = pd.DataFrame({"OR": ..., "2.5 %": ..., "97.5 %": ...})


In [ ]:
#@title ✅ Check — 🚀 A3 side quest { display-mode: "form" }
check('🚀 A3 side quest', lambda: or_table.filter(like="Two year", axis=0)["97.5 %"].iloc[0] < 1 and 0.1 < or_table.filter(like="Two year", axis=0)["OR"].iloc[0] < 0.35, hint='np.exp(logit_res.conf_int()) has two columns (0 and 1): the lower and upper limit; the row is "C(Contract)[T.Two year]".')

## A4 · Decision trees and overfitting (10 min)

A decision tree asks yes/no questions about the features; a shallow tree is a readable set of rules.

In [ ]:
tree3 = Pipeline([("prep", make_telco_preprocess()),
                  ("clf", DecisionTreeClassifier(max_depth=3, random_state=42))]).fit(Xt_train, yt_train)
plt.figure(figsize=(18, 6))
plot_tree(tree3.named_steps["clf"], feature_names=tree3.named_steps["prep"].get_feature_names_out(),
          class_names=["stays", "churns"], filled=True, rounded=True, fontsize=8)
plt.title("A depth-3 tree for Telco churn (colour = majority class, darker = purer)")
plt.show()

Read it from the top: the first question is the contract (`Contract_Month-to-month <= 0.5`), then monthly charges, internet service and tenure — the same story as the odds ratios. In each box, `samples` counts customers and `value` gives the shares that stay and churn.

👉 In Colab, move the slider (or type another number) and rerun the cell: watch training and test AUC as the tree grows.

In [ ]:
max_depth = 6  #@param {type:"slider", min:1, max:25, step:1}
tree = Pipeline([("prep", make_telco_preprocess()),
                 ("clf", DecisionTreeClassifier(max_depth=max_depth, random_state=42))]).fit(Xt_train, yt_train)
print(f"max_depth = {max_depth}:   train AUC {roc_auc_score(yt_train, tree.predict_proba(Xt_train)[:, 1]):.3f}"
      f"   test AUC {roc_auc_score(yt_test, tree.predict_proba(Xt_test)[:, 1]):.3f}")

> 🔮 **Predict before you run:** As `max_depth` goes from 1 to 20, what happens to the **training** AUC and to the **test** AUC?

In [ ]:
rows = []
for d in range(1, 21):
    m = Pipeline([("prep", make_telco_preprocess()),
                  ("clf", DecisionTreeClassifier(max_depth=d, random_state=42))]).fit(Xt_train, yt_train)
    rows.append((d, roc_auc_score(yt_train, m.predict_proba(Xt_train)[:, 1]),
                 roc_auc_score(yt_test, m.predict_proba(Xt_test)[:, 1])))
curve = pd.DataFrame(rows, columns=["depth", "train AUC", "test AUC"]).set_index("depth")

ax = curve.plot(marker="o", figsize=(9, 4))
ax.axvspan(0.5, 3.5, alpha=0.1, color="tab:blue")
ax.axvspan(9.5, 20.5, alpha=0.1, color="tab:red")
ax.text(1.0, 0.62, "underfitting", color="tab:blue")
ax.text(13, 0.62, "overfitting", color="tab:red")
ax.set_xlabel("max_depth")
ax.set_ylabel("ROC-AUC")
ax.set_title("Model complexity vs generalisation (Telco, decision tree)")
plt.show()

**So what?** Shallow trees score low on both sets (underfitting). Deep trees score almost 1.0 on the training data while the test AUC falls (overfitting). The best test AUC (≈ 0.83) lies in between, at depth 5–6.

> ⚠️ **Common mistake:** choosing `max_depth` from a plot like this one. It uses the **test** set, so the test score is no longer an honest estimate for new data. Choose with cross-validation on the training data (next section) and look at the test set **once**, at the end.

## A5 · The generalisation toolkit (15 min)

**k-fold cross-validation** (AI Studio: *Cross Validation*): split the training data into *k* folds; train on *k − 1*, validate on the remaining one; repeat *k* times. You get *k* scores instead of one lucky (or unlucky) split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(Pipeline([("prep", make_telco_preprocess()),
                                   ("clf", DecisionTreeClassifier(max_depth=5, random_state=42))]),
                         Xt_train, yt_train, cv=cv, scoring="roc_auc")
print("AUC per fold:", scores.round(3), "→ mean", round(scores.mean(), 3), "± sd", round(scores.std(), 3))

**So what?** The five fold scores range from 0.81 to 0.85 — a single train/test split could have given any of them. Report the **mean ± sd** (here 0.829 ± 0.011).

> 📈 **Economist's view.** k-fold CV is an out-of-sample robustness check: the model must hold up on k samples it was not fitted on. The spread of the k scores shows how much the result depends on the particular sample — much as a standard error does for a coefficient.

> 💻 **Informatics view.** Cross-validation is the unit-test suite of a model: k independent tests instead of one manual check, rerun after every change. A model that passes only on one lucky split is a flaky test.

In [ ]:
# 🚀 Side quest: How stable is a CV score? One 5-fold CV is one random shuffle of the data.
#    RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42) repeats it with 10 different shuffles.
#    Get the 50 fold scores of the same depth-5 tree pipeline (`rep_scores`, with cross_val_score) and the 10
#    repeat means (`rep_means`, one per shuffle). Plot a histogram of the 50 scores and mark the 10 means.
#    How much more stable is a 5-fold mean than a single fold?
# from sklearn.model_selection import RepeatedStratifiedKFold
# tree5 = Pipeline([...])                              # the same depth-5 tree pipeline as above
# rep_scores = cross_val_score(tree5, Xt_train, yt_train, scoring="roc_auc", cv=RepeatedStratifiedKFold(...))
# rep_means = ...                                      # tip: rep_scores.reshape(10, 5)


In [ ]:
#@title ✅ Check — 🚀 A5 side quest { display-mode: "form" }
check('🚀 A5 side quest', lambda: len(rep_scores) == 50 and len(rep_means) == 10 and rep_means.std() < rep_scores.std() / 3, hint='RepeatedStratifiedKFold yields all 5 folds of shuffle 1, then shuffle 2, …: reshape(10, 5).mean(axis=1).')

**Learning curves** show training and validation scores as the training set grows:
- both curves low and close together → **underfitting** (high bias): use a more flexible model or better features;
- a large gap → **overfitting** (high variance): regularise, simplify or add data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
models = [("logistic regression", LogisticRegression(max_iter=1000)),
          ("decision tree, depth 20", DecisionTreeClassifier(max_depth=20, random_state=42))]
for ax, (name, clf) in zip(axes, models):
    sizes, tr, va = learning_curve(Pipeline([("prep", make_telco_preprocess()), ("clf", clf)]),
                                   Xt_train, yt_train, cv=cv, scoring="roc_auc",
                                   train_sizes=np.linspace(0.1, 1.0, 6))
    ax.plot(sizes, tr.mean(axis=1), marker="o", label="training")
    ax.plot(sizes, va.mean(axis=1), marker="o", label="validation")
    ax.set_title(name)
    ax.set_xlabel("training examples")
    ax.legend()
axes[0].set_ylabel("ROC-AUC")
fig.suptitle("Learning curves on Telco (5-fold CV)")
plt.show()

Logistic regression: the curves meet around 0.85 → little overfitting; more data would hardly help. The depth-20 tree: training 1.0, validation about 0.65 → massive overfitting, and more data does not close the gap. **Limit the complexity.**

**Grid search** tries every combination of hyperparameters with cross-validation. Inside a pipeline, parameters are addressed as `step__parameter` (two underscores).

In [ ]:
param_grid = {"clf__max_depth": [3, 4, 5, 6, 8], "clf__min_samples_leaf": [1, 10, 30, 60]}
grid = GridSearchCV(Pipeline([("prep", make_telco_preprocess()), ("clf", DecisionTreeClassifier(random_state=42))]),
                    param_grid, cv=cv, scoring="roc_auc")
grid.fit(Xt_train, yt_train)
print("best:", grid.best_params_, "| CV AUC:", round(grid.best_score_, 3))
print("test AUC of the tuned tree:", round(roc_auc_score(yt_test, grid.predict_proba(Xt_test)[:, 1]), 3))

**So what?** The search chose depth 5 with at least 60 customers per leaf. Its CV AUC (0.832) and the test AUC (0.829) agree: the tuned tree generalises as cross-validation promised — but it is still below logistic regression (0.842).

In [ ]:
# 🧪 Try it: Tune the regularisation strength C of the logistic-regression pipeline with GridSearchCV
#    (C in [0.01, 0.1, 1, 10], cv=cv, scoring="roc_auc"). Store the fitted search in `grid_lr` and the best C in `best_C`.
#    (Smaller C = stronger penalty.)
# grid_lr = GridSearchCV(Pipeline([...]), {"clf__C": [...]}, cv=cv, scoring="roc_auc").fit(...)
# best_C = ...


In [ ]:
#@title ✅ Check — A5 Try it { display-mode: "form" }
check('A5 Try it', lambda: best_C >= 1 and grid_lr.best_score_ > 0.84, hint='The parameter name inside the pipeline is "clf__C"; read it back from grid_lr.best_params_.')

💡 If the best value sits at the **edge** of your grid (here C = 10), extend the grid in that direction before you trust it. Here C = 100 or 1,000 score the same as C = 10 (differences below 0.0002): the scores have reached a plateau, and any C from 1 upwards is fine. Whether such a tiny difference is real is a question for a statistical test (🚀 C4).

## A6 · Assignment briefing (10 min)

**Homework 3 — "Beat the AutoML" (individual, Python).** You know the data from Homework 1: a Taiwanese bank wants to know which credit-card clients will **default** next month (UCI data, 30,000 clients). You deliver one Colab notebook that runs top to bottom: EDA → data-quality decisions → features → a leak-proof pipeline → several models (a dummy baseline, logistic regression, a tuned tree and a tuned XGBoost from S14), compared with an **AutoGluon** baseline → a decision threshold chosen from the given costs → a memo of at most 250 words to the head of credit risk.
- Beating AutoGluon is welcome but **not required**: a sound process, honest evaluation and a clear business decision earn the points.
- Today's toolkit is exactly what the modelling and evaluation sections need: baselines, pipelines, cross-validation, grid search — and the test set used **once**.
- 10 points, like Homeworks 1 and 2; the rubric (*presojnica*) is attached to Homework 3 on Canvas.
- AI tools are **allowed and must be declared**. You must be able to explain every line of your notebook (short oral check at the project presentations).
- Deadline: **Sunday 17 January 2027, 23:59**, on Canvas.

> **Before the break:** read the Part B scenario below.

---
## ☕ Break — 15 minutes

---
# Part B · Your turn: will the client subscribe? (35 min)

**Scenario.** A Portuguese bank ran telemarketing campaigns for term deposits (Moro, Cortez & Rita, 2014; 45,211 calls). Only 11.7 % of the clients said yes. The bank wants a model that tells the call centre whom to call **before** calling.

In [ ]:
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
outer = zipfile.ZipFile(io.BytesIO(requests.get(BANK_URL, timeout=60).content))   # the download is a zip ...
inner = zipfile.ZipFile(io.BytesIO(outer.read("bank.zip")))                       # ... containing another zip
bank = pd.read_csv(inner.open("bank-full.csv"), sep=";")
print(bank.shape)
bank.head(3)

In [ ]:
# ✏️ TODO: Task 1 — Leakage and split. `duration` (the length of the call) is only known AFTER the call,
#    so drop it. Create `X_bank` (all columns except `y` and `duration`) and `y_bank` (1 = "yes"),
#    then split 80/20 with stratify=y_bank and random_state=42 into Xb_train, Xb_test, yb_train, yb_test.
# X_bank = bank.drop(columns=[...])
# y_bank = ...
# Xb_train, Xb_test, yb_train, yb_test = train_test_split(...)


In [ ]:
#@title ✅ Check — Task 1 { display-mode: "form" }
check('Task 1', lambda: "duration" not in Xb_train.columns and Xb_train.shape == (36168, 15) and abs(yb_train.mean() - 0.117) < 0.005, hint="drop(columns=['y', 'duration']); train_test_split(X_bank, y_bank, test_size=0.2, stratify=y_bank, random_state=42)")

In [ ]:
#@title 🛟 Rescue — run only if you could not finish Task 1 { display-mode: "form" }
X_bank = bank.drop(columns=["y", "duration"])
y_bank = (bank["y"] == "yes").astype(int)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bank, y_bank, test_size=0.2,
                                                        stratify=y_bank, random_state=42)
print("split ready:", Xb_train.shape)

In [ ]:
# ✏️ TODO: Task 2 — The dummy baseline. Fit DummyClassifier(strategy="most_frequent") on the training data
#    and store its accuracy on the test set in `dummy_acc`.
# dummy = DummyClassifier(strategy="most_frequent").fit(...)
# dummy_acc = ...


In [ ]:
#@title ✅ Check — Task 2 { display-mode: "form" }
check('Task 2', lambda: abs(dummy_acc - 0.883) < 0.002, hint='accuracy_score(yb_test, dummy.predict(Xb_test))')

**✍️ Your answer** — The dummy model predicts 'no' for everyone and still reaches about 88 % accuracy. What does this teach you?

> *(double-click this cell and write your answer here)*

In [ ]:
NUM_B = ["age", "balance", "day", "campaign", "pdays", "previous"]
CAT_B = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

def make_bank_preprocess():
    return ColumnTransformer([("num", StandardScaler(), NUM_B),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_B)])
print("numeric:", NUM_B, "\ncategorical:", CAT_B)

In [ ]:
# ✏️ TODO: Task 3 — Logistic regression. Build a pipeline with make_bank_preprocess() and
#    LogisticRegression(max_iter=1000). Store the mean 5-fold cross-validated ROC-AUC on the TRAINING
#    data in `auc_logreg`.
# logreg_bank = Pipeline([...])
# auc_logreg = cross_val_score(..., cv=5, scoring="roc_auc").mean()


In [ ]:
#@title ✅ Check — Task 3 { display-mode: "form" }
check('Task 3', lambda: 0.74 < auc_logreg < 0.79, hint='cross_val_score(logreg_bank, Xb_train, yb_train, cv=5, scoring="roc_auc").mean()')

In [ ]:
#@title 🛟 Rescue — run this if `make_bank_preprocess` is not defined (e.g. after a runtime restart) { display-mode: "form" }
NUM_B = ["age", "balance", "day", "campaign", "pdays", "previous"]
CAT_B = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

def make_bank_preprocess():
    return ColumnTransformer([("num", StandardScaler(), NUM_B),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_B)])
print("make_bank_preprocess() is defined.")

In [ ]:
# ✏️ TODO: Task 4 — Validation curve. With validation_curve, compute the 5-fold CV ROC-AUC of a decision-tree
#    pipeline (make_bank_preprocess() + DecisionTreeClassifier(random_state=42)) for max_depth = 1…15
#    (param_name="clf__max_depth"). Plot training and validation means, and store the best depth in
#    `best_depth` and the mean validation scores in `val_auc`. Be patient: 15 depths × 5 folds = 75 fits,
#    about a minute in Colab.
# depths = list(range(1, 16))
# train_auc, val_scores = validation_curve(..., param_name="clf__max_depth", param_range=depths, cv=5, scoring="roc_auc")
# val_auc = ...
# best_depth = ...


In [ ]:
#@title ✅ Check — Task 4 { display-mode: "form" }
check('Task 4', lambda: 5 <= best_depth <= 9 and val_auc.max() > 0.69, hint='val_scores has one row per depth and one column per fold — average along axis=1.')

In [ ]:
# ✏️ TODO: Task 5 — Grid search. Tune the tree with GridSearchCV over max_depth in [3, 5, 7, 9, None] and
#    min_samples_leaf in [1, 20, 50, 100] (cv=5, scoring="roc_auc"). Store the fitted search in
#    `tree_grid` and its best CV score in `tree_grid_auc`. 20 settings × 5 folds = 100 fits: about a minute.
# tree_grid = GridSearchCV(Pipeline([...]), {...}, cv=5, scoring="roc_auc").fit(Xb_train, yb_train)
# tree_grid_auc = ...


In [ ]:
#@title ✅ Check — Task 5 { display-mode: "form" }
check('Task 5', lambda: 0.74 < tree_grid_auc < 0.79, hint='Parameter names: "clf__max_depth" and "clf__min_samples_leaf".')

**✍️ Your answer** — **Diagnosis.** Compare the dummy, logistic regression, the depth-tuned tree and the grid-searched tree. Where do you see under- or overfitting, and which way of limiting complexity worked best?

> *(double-click this cell and write your answer here)*

In [ ]:
# ✏️ ⭐ Stretch: Regression check-up: on the Energy Efficiency data from S10, predict `heating_load` (Y1) from X1–X8
#    with a pipeline StandardScaler → RidgeCV(alphas=np.logspace(-3, 3, 13)) (80/20 split, random_state=42).
#    Store the test R² in `r2_energy`.
# energy = pd.read_excel(...)
# ...


In [ ]:
#@title ✅ Check — Stretch { display-mode: "form" }
check('Stretch', lambda: r2_energy > 0.88, hint='RidgeCV chooses alpha by (efficient leave-one-out) cross-validation.')

> 🚀 **Done early?** After the wrap-up, **Part C · Advanced track** takes you under the hood: logistic regression from scratch, bias and variance measured by simulation, nested CV, a corrected t-test, calibrated probabilities and tests for ML code. Join the wrap-up first, then continue there.

---
## Wrap-up (10 min)

**Key takeaways**
1. Judge a model on data it has not seen, and compare it with a baseline (dummy, simple model).
2. **Underfitting:** both scores low → more flexible model or better features. **Overfitting:** a large train–validation gap → regularise, simplify, add data.
3. Ridge shrinks, Lasso shrinks *and* selects; trees are restrained with `max_depth` or `min_samples_leaf`.
4. Cross-validation + grid search choose hyperparameters; the test set is used **once**, at the end.
5. Logistic regression gives probabilities and interpretable odds ratios; a shallow tree gives readable rules.

In [ ]:
progress()

### 📋 Cheat sheet — today's essentials

| I want to … | Code |
|---|---|
| fit a model inside a pipeline | `Pipeline([("prep", preprocess), ("reg", LinearRegression())]).fit(X, y)` |
| regression metrics | `mean_absolute_error(y, p) · root_mean_squared_error(y, p) · r2_score(y, p)` |
| a penalised linear model | `Ridge(alpha=10) · Lasso(alpha=0.001) · LassoCV(cv=5)` |
| count the features Lasso kept | `(model.named_steps["reg"].coef_ != 0).sum()` |
| class probabilities (for ROC-AUC) | `model.predict_proba(X)[:, 1]` |
| odds ratios | `np.exp(model.named_steps["clf"].coef_[0])` |
| draw a shallow tree | `plot_tree(tree, feature_names=names, class_names=classes, filled=True)` |
| an honest score: k-fold CV | `cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="roc_auc")` |
| diagnose: learning curve | `learning_curve(pipe, X, y, cv=cv, scoring="roc_auc")` |
| diagnose: validation curve | `validation_curve(pipe, X, y, param_name="clf__max_depth", param_range=range(1, 16), cv=5)` |
| tune inside a pipeline | `GridSearchCV(pipe, {"clf__max_depth": [3, 5, 7]}, cv=5, scoring="roc_auc").fit(X, y)` |
| a baseline to beat | `DummyClassifier(strategy="most_frequent")` |

> 🧩 **For your project:** start every model comparison with a dummy baseline and a logistic regression in a `Pipeline`. Report cross-validated scores as **mean ± sd**, tune with `GridSearchCV` inside the pipeline, and keep the test set for one final check. Show one learning or validation curve to justify the complexity of your final model, and explain it with odds ratios or the rules of a shallow tree. The assignment follows exactly this recipe.

**✍️ Your answer** — **Exit ticket.** (1) A learning curve shows training AUC 0.99 and validation AUC 0.70 — diagnosis and remedy? (2) Why is `duration` not allowed as a feature? (3) What does `clf__max_depth` mean in a grid?

> *(double-click this cell and write your answer here)*

**Next time — S14 (Mon 11 / Wed 13 Jan):** boosting with **XGBoost**, choosing a decision threshold from business costs, explaining models with SHAP — and **AutoML vs you**. Bring your bank-marketing results: you will try to beat them.

## 🧠 Think it through (optional · no code · 10 min)

Questions for thinking rather than coding, e.g. if you are strong in statistics but new to Python. Discuss them in pairs, or write two or three sentences for each. The solutions notebook has model answers.

1. 📈 The churn model gives fibre-optic internet an odds ratio of about 1.9. The marketing director concludes: "So if we move customers from fibre to DSL, churn will fall." What do you answer?
2. 📈 Econometricians choose a specification by theory and p-values; today we chose `max_depth` and `C` by cross-validated AUC. When is each approach the right one?
3. 💻 Your churn pipeline goes into production and is retrained every month on fresh data. What could break silently — and which automated checks would you add?
4. 💻 Why is one `Pipeline` that bundles preprocessing and model better software than a script that scales the data first and fits the model afterwards?

> ✍️ *(double-click this cell and write your answers here)*

---
# Part C · 🚀 Advanced track (optional)

**For:** fast-lane students who finished Part B — or anyone who wants more, at home. Never graded; the assignment and the project do not need it.

**Theme:** Under the hood of models and validation. Build logistic regression yourself, measure bias and variance, find out when a cross-validated score — or a difference between two scores — can be trusted, and test ML code like any other software.

| # | Challenge | Time | Level | Lens |
|---|---|---|---|---|
| **C1** | Logistic regression from scratch | 20 min | 🌶️🌶️🌶️ | 📈💻 |
| **C2** | Bias and variance, measured | 20 min | 🌶️🌶️ | 📈 |
| **C3** | Nested cross-validation | 15 min | 🌶️🌶️ | 📈 |
| **C4** | Is the difference real? A corrected t-test | 20 min | 🌶️🌶️🌶️ | 📈 |
| **C5** | Honest probabilities: calibration | 15 min | 🌶️🌶️ | 📈 |
| **C6** | Tests for ML code | 15 min | 🌶️🌶️ | 💻 |
| **🏁** | Open: one more tool, built by you — e.g. k-fold CV or AdaBoost from scratch | 30+ min | open | 📈💻 |

**Lens:** 📈 more statistics and economics, less code · 💻 more programming and data engineering · 📈💻 both

Each challenge stands on its own. **Run the ⚙️ Part C setup cell first**, then pick any challenge. The usual rules apply: a first own attempt, then Gemini may *explain* or give a *hint*.

Part C uses the Telco data from Part A (the setup cell reloads it after a restart) and simulated data in C2. Each solution runs in under a minute on Colab's free CPU; C4, the slowest, takes about half a minute.

In [ ]:
#@title ⚙️ Part C setup — run this first (rebuilds everything Part C needs) { display-mode: "form" }
if "check" not in globals():  # runtime restarted? then restore the ✅ self-check helpers
    CHECK_RESULTS = {}  # label -> True (correct) / False (not yet) / None (not attempted)

    def check(label, test, hint=""):
        """Give instant feedback on an exercise without ever stopping 'Run all'.

        `test` is a function (usually a lambda) that returns True when the answer is right.
        """
        try:
            ok = bool(test())
        except NameError as err:
            CHECK_RESULTS[label] = None
            print(f"⏳ {label}: not attempted yet ({err}).")
            return
        except Exception as err:  # wrong type, wrong shape, missing column, ...
            CHECK_RESULTS[label] = False
            print(f"❌ {label}: checking your answer raised {type(err).__name__}: {err}")
            if hint:
                print(f"   💡 Hint: {hint}")
            return
        CHECK_RESULTS[label] = ok
        if ok:
            print(f"✅ {label}: correct!")
        else:
            print(f"❌ {label}: not yet.")
            if hint:
                print(f"   💡 Hint: {hint}")

    def progress(*, return_count=False):
        """Summarise all self-checks run so far; optional 🚀 fast-lane checks are counted separately."""
        core = [v for label, v in CHECK_RESULTS.items() if not label.startswith("🚀")]
        fast = [v for label, v in CHECK_RESULTS.items() if label.startswith("🚀")]
        solved = sum(1 for v in core if v)
        print(f"Solved {solved} of {len(core)} exercises checked so far.")
        if fast:
            print(f"🚀 Fast lane (optional): solved {sum(1 for v in fast if v)} of {len(fast)}.")
        for label, v in CHECK_RESULTS.items():
            print(f"  {'✅' if v else ('⏳' if v is None else '❌')} {label}")
        if return_count:
            return solved

    print("Helpers loaded ✔")

import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

if "Xt_train" not in globals():           # the Telco data, split and preprocessing from A3
    TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    telco = pd.read_csv(TELCO_URL)
    telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)   # S10: not billed yet
    y_tel = (telco["Churn"] == "Yes").astype(int)
    X_tel = telco.drop(columns=["customerID", "Churn"])

    NUM_TEL = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
    CAT_TEL = [c for c in X_tel.columns if c not in NUM_TEL]
    Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_tel, y_tel, test_size=0.2, stratify=y_tel, random_state=42)

    def make_telco_preprocess():
        return ColumnTransformer([("num", StandardScaler(), NUM_TEL),
                                  ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_TEL)])

# C1 · a small, well-conditioned design: 4 numeric features + 5 dummies, all standardised
C1_DATA = pd.concat([Xt_train[NUM_TEL],
                     pd.get_dummies(Xt_train[["Contract", "InternetService", "PaperlessBilling"]],
                                    drop_first=True, dtype=float)], axis=1)
C1_COLS = list(C1_DATA.columns)
Xc1, yc1 = StandardScaler().fit_transform(C1_DATA), yt_train.to_numpy()

# C2 · the truth behind the simulation
def f_true(x):
    return np.sin(2 * np.pi * x)

SIGMA = 0.3

# C3 and 🏁 · Telco as one numeric table for trees: no scaling needed, and 0/1 dummy columns learn no
# statistics from the data, so encoding before the split cannot leak
X_enc = pd.get_dummies(Xt_train, dtype=float)
X_enc_test = pd.get_dummies(Xt_test, dtype=float).reindex(columns=X_enc.columns, fill_value=0)

# C3 · 300 customers whose churn labels C3 shuffles: then no model can beat ROC-AUC 0.5
X_c3, _, y_c3, _ = train_test_split(X_enc, yt_train, train_size=300, stratify=yt_train, random_state=0)
X_c3, y_c3 = X_c3.to_numpy(), y_c3.to_numpy()
C3_GRID = {"max_depth": [2, 3, 4, 6, None], "min_samples_leaf": [1, 5, 10, 20]}

# C6 · the A3 pipeline as a factory, and a copy with three hidden bugs (find them with tests, not by reading this)
def make_telco_pipeline():
    return Pipeline([("prep", make_telco_preprocess()), ("clf", LogisticRegression(max_iter=1000))])

_SCALER_ALL_ROWS = StandardScaler().fit(X_tel[NUM_TEL])

def make_broken_pipeline():
    return Pipeline([("prep", ColumnTransformer([("num", FunctionTransformer(_SCALER_ALL_ROWS.transform), NUM_TEL),
                                                 ("cat", OneHotEncoder(handle_unknown="error"), CAT_TEL)])),
                     ("clf", RandomForestClassifier(n_estimators=20))])

print("Part C ready ✔  C1:", Xc1.shape, "| X_enc:", X_enc.shape)

## C1 · Logistic regression from scratch 🌶️🌶️🌶️ (20 min) · 📈💻

**Why it matters.** `LogisticRegression` is a loss function, a penalty and an optimiser — nothing more. Build it once, and `C`, the unpenalised intercept and the solver's `tol` stop being magic.

**What to build.** A function `fit_logreg_gd(X, y, C=1.0, lr=1.0, n_iter=5000)` that returns `(b, w, history)`: plain gradient descent, vectorised (no loop over rows), on the objective scikit-learn minimises:

$$J(b, w) = \frac{1}{n}\sum_{i=1}^{n}\Big[-y_i\log p_i-(1-y_i)\log(1-p_i)\Big] + \frac{\lVert w\rVert^2}{2Cn}, \qquad p_i = \frac{1}{1+e^{-(b+x_i^\top w)}}$$

scikit-learn writes it as ½‖w‖² + C · Σ log-loss — the same minimum, multiplied by C·n. The intercept *b* is **not** penalised. Run it on `Xc1`, `yc1` (from the Part C setup: nine standardised Telco features, names in `C1_COLS`) and store `b_gd`, `w_gd`.

**Hints.** ∇w J = Xᵀ(p − y)/n + w/(C·n) and ∂J/∂b = mean(p − y). With standardised features, `lr=1` and about 5,000 steps are enough. Record J every 500 steps — it must go down. Unsure about your gradient? Compare it with a finite difference.

**Check.** Your coefficients must match a *tightly* converged `LogisticRegression(C=1.0, tol=1e-10)` to four decimals. Bonus: compare with the default fit, too — it stops at `tol=1e-4`, about 0.0008 away from the optimum.

In [ ]:
# 🚀 Challenge: Logistic regression by gradient descent: write fit_logreg_gd(X, y, C, lr, n_iter)
#    → (b, w, history), run it on Xc1, yc1 and store b_gd, w_gd. Compare with LogisticRegression(C=1.0).
# def sigmoid(z):
#     return 1 / (1 + np.exp(-z))
#
# def fit_logreg_gd(X, y, C=1.0, lr=1.0, n_iter=5000):
#     n, p = X.shape
#     b, w, history = 0.0, np.zeros(p), []
#     for step in range(n_iter):
#         prob = ...
#         grad_w = ...
#         grad_b = ...
#         ...
#     return b, w, history
#
# b_gd, w_gd, history = fit_logreg_gd(Xc1, yc1)


In [ ]:
#@title ✅ Check — 🚀 C1 { display-mode: "form" }
def _check_c1():
    mine = np.r_[b_gd, w_gd]
    ref = LogisticRegression(C=1.0, tol=1e-10, max_iter=10_000).fit(Xc1, yc1)
    return mine.shape == (10,) and np.allclose(mine, np.r_[ref.intercept_, ref.coef_[0]], atol=1e-4)

check("🚀 C1", _check_c1,
      hint="Penalty term w / (C·n) in the gradient, none for b; about 5,000 steps with lr=1. Check the gradient with a finite difference.")

## C2 · Bias and variance, measured 🌶️🌶️ (20 min) · 📈

**Why it matters.** "Shallow trees have high bias, deep trees high variance" is usually drawn as a cartoon. With simulated data — where we *know* the truth — you can measure both.

**What to build.** The truth is f(x) = sin(2πx) on [0, 1], plus noise with σ = 0.3 (`f_true` and `SIGMA` from the setup). With `rng = np.random.default_rng(42)`, draw **200 training sets of 100 points** each. For every depth d = 1 … 10, fit a `DecisionTreeRegressor(max_depth=d, random_state=0)` on each set and predict on 200 fixed test points `x0 = np.linspace(0, 1, 200)`. Then, per depth:
- **bias²** = mean over x0 of (average prediction − f(x0))²;
- **variance** = mean over x0 of the variance of the 200 predictions;
- **total** = mean squared error against fresh noisy targets y0 = f(x0) + noise (new noise for every training set).

Store a DataFrame `bv` indexed by depth (1–10) with the columns `bias2`, `variance` and `total`, and plot the three curves.

**Check.** Bias² falls with depth, variance grows, the best depth lies in between, and total ≈ bias² + variance + σ² — the famous decomposition.

**Hints.** Draw all training sets at once, e.g. `x_sets = rng.uniform(0, 1, size=(200, 100))`. Keep the predictions of one depth in an array of shape (200 sets, 200 points); then `.mean(axis=0)` and `.var(axis=0)` do the rest.

In [ ]:
# 🚀 Challenge: Bias² / variance / total error vs tree depth by simulation: build `bv` (index: depth 1–10;
#    columns bias2, variance, total) and plot it.
# rng = np.random.default_rng(42)
# N_SETS, N_TRAIN = 200, 100
# x0 = np.linspace(0, 1, 200)                         # fixed test points
# x_sets = rng.uniform(0, 1, size=(N_SETS, N_TRAIN))
# y_sets = ...                                        # f_true + noise
# y0 = ...                                            # fresh noisy targets at x0, one row per training set
# rows = []
# for depth in range(1, 11):
#     preds = ...                                     # shape (N_SETS, 200): one row per training set
#     rows.append({"depth": depth, "bias2": ..., "variance": ..., "total": ...})
# bv = pd.DataFrame(rows).set_index("depth")


In [ ]:
#@title ✅ Check — 🚀 C2 { display-mode: "form" }
check('🚀 C2', lambda: bv.loc[1, "bias2"] > 20 * bv.loc[10, "bias2"] and bv.loc[10, "variance"] > 2 * bv.loc[1, "variance"] and 2 <= bv["total"].idxmin() <= 5 and np.allclose(bv["total"], bv["bias2"] + bv["variance"] + SIGMA ** 2, rtol=0.05), hint='total uses FRESH noisy targets y0 (not f(x0)); bias² uses the average prediction over the 200 sets.')

**✍️ Your answer** — At the best depth, which part of the error is the largest — and which part could a better method still reduce?

> *(double-click this cell and write your answer here)*

## C3 · Nested cross-validation 🌶️🌶️ (15 min) · 📈

**Why it matters.** `GridSearchCV.best_score_` is the **maximum** of many noisy CV scores, and the maximum of noisy numbers is biased upwards: the winning setting wins partly by luck. Report it as your model's performance and you oversell. **Nested CV** runs the whole search inside an outer CV loop, so every outer test fold is untouched by the tuning.

**What to build.** We make the bias impossible to miss with the classic trick of Varma & Simon (2006): data **without any signal**. `X_c3` / `y_c3` (Part C setup) are 300 Telco customers; shuffle their churn labels, and no model can beat ROC-AUC 0.5 — that is the truth. For each of **3 shuffles** (`rng = np.random.default_rng(0)`, then `y_noise = rng.permutation(y_c3)`):
1. tune `DecisionTreeClassifier(random_state=0)` with `GridSearchCV` over `C3_GRID` (20 settings; inner CV `StratifiedKFold(5, shuffle=True, random_state=1)`, `scoring="roc_auc"`) and keep its `best_score_`;
2. run nested CV: `cross_val_score(search, X_c3, y_noise, cv=outer_cv, scoring="roc_auc")` with the same (unfitted) search and `outer_cv = StratifiedKFold(5, shuffle=True, random_state=2)`, and keep the mean.

Store the averages over the three shuffles as `best_c3` and `nested_c3`.

**Check.** The search "finds" signal that is not there (`best_c3` clearly above 0.5); nested CV stays close to the truth.

In [ ]:
# 🚀 Challenge: Nested CV on data without signal: average best_score_ (best_c3) and the nested CV estimate
#    (nested_c3) over 3 label shuffles. The truth is ROC-AUC 0.5.
# rng = np.random.default_rng(0)
# inner_cv = StratifiedKFold(5, shuffle=True, random_state=1)
# outer_cv = StratifiedKFold(5, shuffle=True, random_state=2)
# best_scores, nested_scores = [], []
# for r in range(3):
#     y_noise = rng.permutation(y_c3)
#     search = GridSearchCV(...)
#     best_scores.append(...)               # best_score_ after search.fit(X_c3, y_noise)
#     nested_scores.append(...)             # cross_val_score(search, ..., cv=outer_cv).mean()
# best_c3, nested_c3 = ..., ...


In [ ]:
#@title ✅ Check — 🚀 C3 { display-mode: "form" }
check('🚀 C3', lambda: best_c3 > 0.51 and best_c3 > nested_c3 + 0.015 and abs(nested_c3 - 0.5) < 0.05, hint='Fit the search on (X_c3, y_noise) for best_score_; for nested CV pass the search itself to cross_val_score with cv=outer_cv.')

**✍️ Your answer** — Why is `best_score_` optimistic — and which number do you report in a project?

> *(double-click this cell and write your answer here)*

## C4 · Is the difference real? A corrected t-test 🌶️🌶️🌶️ (20 min) · 📈

**Why it matters.** In A5 the grid search preferred C = 10 over C = 1 by 0.0002 AUC. Real, or noise? A paired t-test on cross-validation scores looks like the answer — but the folds share most of their training data, so the scores are **not independent**, and the naive test is far too confident. Nadeau and Bengio (2003) corrected the variance for that overlap.

**What to build.** With `RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=42)` — J = 100 splits of the Telco training data — fit the A3 logistic-regression pipeline with **C = 10** and with **C = 1** on the *same* training folds. Store the 100 paired differences of the validation ROC-AUC (C = 10 minus C = 1) in `diffs_c4`. From their mean d̄ and sample variance s² (`ddof=1`), compute two t-statistics with J − 1 degrees of freedom:

| Test | Statistic |
|---|---|
| naive paired t-test | t = d̄ / √(s² / J) |
| Nadeau–Bengio corrected | t = d̄ / √((1/J + n_test/n_train) · s²) |

For 10-fold CV, n_test/n_train = 1/9. Store `t_naive`, `t_nb` and the two-sided p-values `p_naive`, `p_nb` (`2 * stats.t.sf(abs(t), J - 1)`).

**Hints.** Fit the preprocessing once per fold and reuse it for both values of C: that halves the run time and is still leak-free. Expect about half a minute.

**Check.** Your statistics match the formulas — and the two tests disagree.

In [ ]:
# 🚀 Challenge: Paired comparison of C = 10 vs C = 1 on 10 × 10-fold CV: diffs_c4, then t_naive / p_naive and
#    the Nadeau–Bengio corrected t_nb / p_nb.
# rcv = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=42)
# diffs_c4 = []
# for train_idx, val_idx in rcv.split(Xt_train, yt_train):
#     ...                  # both models: the same training fold, the same validation fold
#     diffs_c4.append(auc_c10 - auc_c1)
# diffs_c4 = np.array(diffs_c4)
# J = len(diffs_c4)
# t_naive = ...
# t_nb = ...
# p_naive, p_nb = ...


In [ ]:
#@title ✅ Check — 🚀 C4 { display-mode: "form" }
def _check_c4():
    d = np.asarray(diffs_c4, dtype=float)
    J, s2 = len(d), d.var(ddof=1)
    t_n = d.mean() / np.sqrt(s2 / J)
    t_c = d.mean() / np.sqrt((1 / J + 1 / 9) * s2)
    return (J == 100 and np.isclose(t_naive, t_n, rtol=1e-3) and np.isclose(t_nb, t_c, rtol=1e-2)
            and np.isclose(p_nb, 2 * stats.t.sf(abs(t_c), J - 1), rtol=0.05) and p_naive < 0.05 < p_nb)

check("🚀 C4", _check_c4,
      hint="s² = diffs_c4.var(ddof=1); J = 100; the correction adds n_test/n_train = 1/9 to 1/J. Same folds for both models.")

**✍️ Your answer** — What do you conclude about C = 10 vs C = 1 — and why do the two tests disagree?

> *(double-click this cell and write your answer here)*

## C5 · Honest probabilities: calibration 🌶️🌶️ (15 min) · 📈

**Why it matters.** A model can rank customers well and still state the wrong probabilities. If it says 0.3, about 30 % of those customers should churn — otherwise every cost-based threshold (S14, the assignment) is built on sand.

**What to build.** On the Telco split from Part A, fit (1) the logistic-regression pipeline, (2) a deep tree — `DecisionTreeClassifier(random_state=42)` without a depth limit — in the same preprocessing pipeline, and (3) that deep-tree pipeline wrapped in `CalibratedClassifierCV(..., method="isotonic", cv=5)`. On the **test** set:
- draw the three **calibration curves**: `calibration_curve(y_true, p, n_bins=10)` returns, per bin, the observed churn rate and the mean predicted probability; add the diagonal (perfect calibration);
- compute the **Brier score** — the mean squared error of the probabilities, `brier_score_loss(y_true, p)` — as `brier_lr`, `brier_tree` and `brier_cal`, plus `brier_base` for the know-nothing forecast that gives every customer the training churn rate.

**Check.** Brier scores: logistic regression < calibrated tree < deep tree — and the deep tree loses even to the know-nothing forecast.

In [ ]:
# 🚀 Challenge: Calibration curves and Brier scores on the Telco test set: logistic regression, a deep
#    tree and the calibrated deep tree. Store brier_lr, brier_tree, brier_cal and brier_base.
# lr_c5 = Pipeline([("prep", make_telco_preprocess()), ("clf", LogisticRegression(max_iter=1000))]).fit(Xt_train, yt_train)
# tree_c5 = ...
# cal_c5 = CalibratedClassifierCV(..., method="isotonic", cv=5).fit(Xt_train, yt_train)
# probs = {"logistic regression": lr_c5.predict_proba(Xt_test)[:, 1], ...}
# for name, p in probs.items():
#     frac_pos, mean_pred = calibration_curve(yt_test, p, n_bins=10)
#     ...
# brier_lr, brier_tree, brier_cal = ...
# brier_base = ...


In [ ]:
#@title ✅ Check — 🚀 C5 { display-mode: "form" }
check('🚀 C5', lambda: brier_lr < 0.15 and brier_lr < brier_cal < brier_tree and brier_tree > brier_base, hint='brier_score_loss(yt_test, p) with p = predict_proba(Xt_test)[:, 1]; brier_base uses yt_train.mean() for everyone.')

**✍️ Your answer** — Why does the deep tree lose even to the know-nothing forecast — and what did calibration fix, and what not?

> *(double-click this cell and write your answer here)*

## C6 · Tests for ML code 🌶️🌶️ (15 min) · 💻

**Why it matters.** ML code fails silently: a scaler fitted on the wrong rows, a missing seed or a category that first appears in production raises no error — it just gives wrong numbers. A few `assert`-based tests catch such bugs before your users do.

**What to build.** Four test functions. Each takes a pipeline *factory* (a function that returns a fresh, unfitted pipeline) and checks one property with `assert`:
1. **`no_leakage`** — fitted on `Xt_train`, the preprocessing maps the training rows to numeric columns with mean 0 and SD 1. A scaler fitted on exactly these rows does that; one fitted on other rows does not.
2. **`deterministic`** — two pipelines fitted from scratch give identical test predictions.
3. **`unseen_category`** — test rows with a category never seen in training (`PaymentMethod = "Bitcoin"`) still get probabilities between 0 and 1.
4. **`shuffled_labels`** — shuffle the churn labels of *all* customers (`y_tel`, with `np.random.default_rng(0)`), keep the same training and test rows, train on the shuffled training labels and score against the shuffled test labels. With no signal left, ROC-AUC must be ≈ 0.5 (|AUC − 0.5| < 0.05): this tests your train-and-evaluate code itself.

Then write `run_ml_tests(make_pipe)`: it runs the four tests and returns a report dict `{test name: True / False}`, where a test that raises anything (a failed `assert` or a crash) counts as failed. Run it on `make_telco_pipeline` (the A3 pipeline) and on `make_broken_pipeline`, which the setup cell built with three hidden bugs.

**Hints.** Wrap each test call in `try` / `except Exception`. For test 1, transform `Xt_train` with the fitted `prep` step; the numeric block comes first. For test 3, use `Xt_test.head(5).assign(PaymentMethod="Bitcoin")`.

**Check.** All four tests pass for the A3 pipeline — and together they catch all three bugs in the broken one.

In [ ]:
# 🚀 Challenge: Tests for ML code: four assert-based tests and run_ml_tests(make_pipe) → {test name: True/False};
#    run it on make_telco_pipeline and on make_broken_pipeline.
# def test_no_leakage(make_pipe):
#     pipe = make_pipe().fit(Xt_train, yt_train)
#     Z = pipe.named_steps["prep"].transform(Xt_train)[:, :len(NUM_TEL)]   # the numeric block comes first
#     assert ..., "numeric features are not standardised on the training rows"
#
# def test_deterministic(make_pipe): ...
# def test_unseen_category(make_pipe): ...
# def test_shuffled_labels(make_pipe): ...
#
# ML_TESTS = {"no_leakage": test_no_leakage, "deterministic": test_deterministic,
#             "unseen_category": test_unseen_category, "shuffled_labels": test_shuffled_labels}
#
# def run_ml_tests(make_pipe):
#     report = {}
#     for name, test in ML_TESTS.items():
#         try:
#             ...
#         except Exception:
#             ...
#     return report
#
# pd.DataFrame({"A3 pipeline": run_ml_tests(make_telco_pipeline),
#               "broken pipeline": run_ml_tests(make_broken_pipeline)})


In [ ]:
#@title ✅ Check — 🚀 C6 { display-mode: "form" }
def _check_c6():
    good, bad = run_ml_tests(make_telco_pipeline), run_ml_tests(make_broken_pipeline)
    names = {"no_leakage", "deterministic", "unseen_category", "shuffled_labels"}
    return (set(good) == names and all(good.values()) and set(bad) == names
            and not (bad["no_leakage"] or bad["deterministic"] or bad["unseen_category"]))

check("🚀 C6", _check_c6,
      hint="Use exactly the four test names; a test that raises counts as failed. All four must pass for "
           "make_telco_pipeline, and the broken pipeline must fail three of them.")

**✍️ Your answer** — Which of the three bugs would you most expect in a real project — and which test would you add next?

> *(double-click this cell and write your answer here)*

## 🏁 Open challenge · One more tool, built by you (30+ min) · 📈💻

Pick one — or invent your own — and verify it against scikit-learn:
- **k-fold CV from scratch:** `my_cross_val_score(model, X, y, k=5)` with your own stratified folds; with the same folds it must reproduce `cross_val_score` exactly.
- **AdaBoost from scratch:** 100 decision stumps with SAMME re-weighting; compare with `AdaBoostClassifier` (the solutions show an example).
- **Pruning instead of `max_depth`:** `cost_complexity_pruning_path` plus cross-validation over `ccp_alpha` — does pruning beat the depth-tuned tree?
- **Bias–variance for polynomials:** repeat C2 with `PolynomialFeatures(degree)` + `LinearRegression` instead of a tree.

Then write in the answer cell what you built, how you checked it, and what you learned.

In [ ]:
# 🏁 Open challenge: your code here (add as many cells as you need)


**✍️ Your answer** — 🏁 What did you build, how did you verify it, and what did you learn?

> *(double-click this cell and write your answer here)*

In [ ]:
progress()

---
## Going further (optional, self-study)

- `PolynomialFeatures` + `Ridge`: watch bias turn into variance as the degree grows.
- `ElasticNet` mixes L1 and L2; `LogisticRegressionCV` tunes C automatically.
- **Nested cross-validation** — an unbiased estimate when you tune *and* evaluate on the same data (🚀 C3).
- Cost-complexity pruning of trees: `DecisionTreeClassifier(ccp_alpha=...)`.
- Read: James, Witten, Hastie, Tibshirani & Taylor, *An Introduction to Statistical Learning with Applications in Python* (2023), ch. 3–6 and 8 — free at statlearning.com.

**Data sources and licences:** Ames Housing — De Cock (2011), via OpenML · Telco Customer Churn — IBM sample data · Bank Marketing — Moro, Cortez & Rita (2014), UCI, CC BY 4.0 · Energy Efficiency — Tsanas & Xifara (2012), UCI, CC BY 4.0.